# TCC1: BERTimbau e few-shot nos dados reais

**Antes de começar**
1. No Google Drive, crie a pasta **`TCC_colab`** em *Meu Drive* e coloque nela o arquivo `dist/colab_pacote.zip`.
2. Aqui no Colab, em **Ambiente de execução > Alterar o tipo de ambiente de execução**, escolha **GPU T4**.
   Se aparecer *"Não foi possível atribuir um back-end"*, escolha **CPU** e salve: o notebook se adapta sozinho.
3. Use **Ambiente de execução > Executar tudo** e autorize o acesso ao Drive.

Tudo o que termina é salvo em `Meu Drive/TCC_colab`. Se a sessão cair, basta executar tudo de novo:
o notebook pula o que já foi feito. O resultado final é `Meu Drive/TCC_colab/resultados_colab.zip`.

| Modo | BERTimbau | Few-shot (modelo aberto, sem chave de API) | Tempo estimado |
|---|---|---|---|
| GPU | 5 sementes, qualificação e objeções | Qwen 2.5 7B, qualificação e objeções | 30 a 60 min |
| CPU | 3 sementes, só qualificação | Qwen 2.5 3B, só qualificação | 2 a 4 h |

Mantenha esta aba aberta enquanto roda.

## 1. Configuração e Google Drive

In [ ]:
# ===== Configuração (detecta GPU sozinho) =====
import os
BASE = '/content'
PROJ = '/content/tcc'
DRIVE = '/content/drive/MyDrive/TCC_colab'
try:
    import torch
    GPU = torch.cuda.is_available()
except Exception:
    GPU = False
if GPU:
    MODELO_LLM = 'qwen2.5:7b'
    SEMENTES = [0, 1, 2, 3, 4]
    TAREFAS = ['qualificacao', 'objecao']
else:
    MODELO_LLM = 'qwen2.5:3b'     # CPU: modelo menor, só a tarefa principal
    SEMENTES = [0, 1, 2]
    TAREFAS = ['qualificacao']
EXTRA_BERT = ''
LLM_ARGS = f'--provider ollama --model {MODELO_LLM}'
print('GPU' if GPU else 'SEM GPU (modo CPU, mais lento)', '| LLM:', MODELO_LLM, '| sementes:', SEMENTES, '| tarefas:', TAREFAS)

from google.colab import drive
drive.mount('/content/drive')

## 2. Pacote com o código e o banco

In [ ]:
import os, shutil, zipfile
os.makedirs(DRIVE, exist_ok=True)
pacote = os.path.join(DRIVE, 'colab_pacote.zip')
if not os.path.exists(pacote):
    from google.colab import files
    print('Não achei Meu Drive/TCC_colab/colab_pacote.zip. Envie o arquivo agora:')
    up = files.upload()
    shutil.copy(next(iter(up)), pacote)
zipfile.ZipFile(pacote).extractall(PROJ)
assert os.path.exists(os.path.join(PROJ, 'data/processed/tcc.db')), 'O pacote precisa conter o banco'
print('pacote extraído')

## 3. Dependências

In [ ]:
!pip -q install pandas scikit-learn scipy tabulate transformers

## 4. Funções auxiliares

In [ ]:
# ===== Funções auxiliares: tudo que termina é salvo no Drive =====
import glob, shutil, subprocess, time, zipfile
os.makedirs(DRIVE, exist_ok=True)
os.chdir(PROJ)
os.environ['PYTHONPATH'] = os.path.join(PROJ, 'src')
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
LOG = os.path.join(DRIVE, 'log_experimentos.txt')

def rodar(cmd):
    t0 = time.time()
    print('>>', cmd, flush=True)
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n>> {cmd}\n')
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for linha in p.stdout:
            print(linha, end='', flush=True)
            log.write(linha); log.flush()
        codigo = p.wait()
        log.write(f'(modo={"GPU" if GPU else "CPU"}, {time.time() - t0:.0f} s, saída={codigo})\n')
    if codigo != 0:
        raise RuntimeError(f'Falhou: {cmd}')

def bert_semente(tarefa, s):
    destino = os.path.join(DRIVE, f'bert_{tarefa}', f'seed{s}', f'preds_bertimbau_{tarefa}.csv')
    if os.path.exists(destino):
        print('já feito:', destino); return
    tmp = os.path.join(BASE, f'tmp_bert_{tarefa}_{s}')
    rodar(f'python -m tcc.experiments.bert --db data/processed/tcc.db --task {tarefa} --seeds {s} --out {tmp} {EXTRA_BERT}')
    arq = glob.glob(os.path.join(tmp, f'preds_bertimbau*_{tarefa}.csv'))[0]
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    shutil.copy(arq, destino)

def few_shot(tarefa):
    feito = glob.glob(os.path.join(DRIVE, f'preds_llm_fewshot_k3_*_{tarefa}.csv'))
    if feito:
        print('já feito:', feito[0]); return
    cache_local = os.path.join(BASE, 'llm_cache.sqlite')
    cache_drive = os.path.join(DRIVE, 'llm_cache.sqlite')
    if os.path.exists(cache_drive) and not os.path.exists(cache_local):
        shutil.copy(cache_drive, cache_local)
    tmp = os.path.join(BASE, 'tmp_llm')
    for limite in (5, 10, 15, 20, 25, 30, None):     # em partes: se cair, o cache no Drive guarda o que já foi feito
        lim = f' --limit {limite}' if limite else ''
        rodar(f'python -m tcc.experiments.llm --db data/processed/tcc.db --task {tarefa} {LLM_ARGS} --k 3 '
              f'--workers 1 --out {tmp} --cache {cache_local}{lim}')
        shutil.copy(cache_local, cache_drive)
    arq = glob.glob(os.path.join(tmp, f'preds_llm_fewshot_k3_*_{tarefa}.csv'))[0]
    shutil.copy(arq, os.path.join(DRIVE, os.path.basename(arq)))

## 5. BERTimbau: qualificação, primeira semente

In [ ]:
for tarefa in TAREFAS[:1]:
    bert_semente(tarefa, SEMENTES[0])

## 6. Instalar o Ollama e baixar o modelo aberto

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request
servidor = subprocess.Popen(['ollama', 'serve'], env=dict(os.environ, OLLAMA_KEEP_ALIVE='120m'),
                            stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('O Ollama não iniciou: veja /content/ollama.log')
rodar(f'ollama pull {MODELO_LLM}')

## 7. Few-shot: qualificação

In [ ]:
few_shot('qualificacao')

## 8. BERTimbau: qualificação, demais sementes

In [ ]:
for tarefa in TAREFAS[:1]:
    for s in SEMENTES[1:]:
        bert_semente(tarefa, s)

## 9. Objeções (só com GPU)

In [ ]:
if 'objecao' in TAREFAS:
    for s in SEMENTES:
        bert_semente('objecao', s)
    few_shot('objecao')
else:
    print('Modo CPU: objeções ficam de fora (só com GPU).')

## 10. Juntar os resultados

In [ ]:
import csv
saida = os.path.join(BASE, 'resultados_colab')
os.makedirs(saida, exist_ok=True)
for tarefa in ['qualificacao', 'objecao']:
    arqs = sorted(glob.glob(os.path.join(DRIVE, f'bert_{tarefa}', 'seed*', f'preds_bertimbau*_{tarefa}.csv')))
    if not arqs:
        continue
    linhas, cab = [], None
    for a in arqs:
        with open(a, newline='', encoding='utf-8') as fh:
            r = csv.DictReader(fh); cab = r.fieldnames; linhas += list(r)
    nome = os.path.basename(arqs[0])
    with open(os.path.join(saida, nome), 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=cab); w.writeheader(); w.writerows(linhas)
    print(nome, len(arqs), 'sementes,', len(linhas), 'linhas')
for a in glob.glob(os.path.join(DRIVE, 'preds_llm_*.csv')) + [LOG]:
    shutil.copy(a, saida); print(os.path.basename(a))
zip_final = os.path.join(DRIVE, 'resultados_colab.zip')
with zipfile.ZipFile(zip_final, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in sorted(glob.glob(os.path.join(saida, '*'))):
        z.write(a, os.path.basename(a))
print('Salvo no Drive:', zip_final)

from google.colab import files
files.download(zip_final)

## 11. Limpar

Depois de baixar `resultados_colab.zip`, apague do Drive a pasta `TCC_colab` (ela contém o banco pseudonimizado).
A célula abaixo apaga a cópia do Colab; ela está comentada para não rodar no "Executar tudo".

In [ ]:
# shutil.rmtree(PROJ, ignore_errors=True)
# print('limpo')